# LeetCode 1460 — Make Two Arrays Equal by Reversing Subarrays bằng Python

**Link đề:** https://leetcode.com/problems/make-two-arrays-equal-by-reversing-subarrays/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting

```python
class Solution:
    def canBeEqual(self, target: list[int], arr: list[int]) -> bool:
        ...
```

### Mục tiêu học
1. Hiểu thao tác **đảo ngược mảng con** (reverse subarray) bằng slicing `a[i:j+1] = a[i:j+1][::-1]`.
2. Nhận ra **bất biến** (invariant) của thao tác: đảo ngược không thay đổi **tập phần tử kèm số lần xuất hiện** (multiset).
3. Chứng minh chiều ngược lại bằng **cách xây dựng**: đảo đoạn độ dài 2 = **hoán đổi hai phần tử kề nhau** → sắp xếp được mọi thứ tự.
4. Đi qua 4 lời giải: mô phỏng xây dựng → sắp xếp rồi so sánh → `Counter` → mảng đếm cố định.
5. Trace, Time/Space, lỗi hay gặp, biến thể (*Valid Anagram*, chỉ được đảo đoạn độ dài cố định…).

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho hai mảng số nguyên **cùng độ dài** `target` và `arr`. Mỗi bước, bạn chọn **một mảng con khác rỗng** (đoạn liên tiếp) của `arr` và **đảo ngược** nó. Được làm **bao nhiêu bước cũng được**.

Trả `true` nếu có thể biến `arr` thành `target`, ngược lại `false`.

| Ví dụ | `target` | `arr` | Output |
|---|---|---|---|
| 1 | `[1,2,3,4]` | `[2,4,1,3]` | `true` |
| 2 | `[7]` | `[7]` | `true` |
| 3 | `[3,7,9]` | `[3,7,11]` | `false` |

Ví dụ 1 (một cách): đảo `[2,4,1]` → `[1,4,2,3]`; đảo `[4,2]` → `[1,2,4,3]`; đảo `[4,3]` → `[1,2,3,4]`.

**Constraints**
- `target.length == arr.length`
- `1 <= target.length <= 1000`
- `1 <= target[i], arr[i] <= 1000`

Câu hỏi gợi mở:
- Thao tác đảo có thể **tạo ra** giá trị mới (như `9` ở ví dụ 3) không? Có làm **mất** giá trị nào không?
- Nếu chỉ được đảo đoạn **độ dài 2**, ta làm được gì?

In [1]:
arr = [2, 4, 1, 3]
print("bắt đầu:", arr)
arr[0:3] = arr[0:3][::-1]; print("đảo [0..2]:", arr)
arr[1:3] = arr[1:3][::-1]; print("đảo [1..2]:", arr)
arr[2:4] = arr[2:4][::-1]; print("đảo [2..3]:", arr)

bắt đầu: [2, 4, 1, 3]
đảo [0..2]: [1, 4, 2, 3]
đảo [1..2]: [1, 2, 4, 3]
đảo [2..3]: [1, 2, 3, 4]


## 2. Nền tảng Python cần dùng

### 2.1. Slicing và đảo ngược đoạn con

- `a[i:j]` là đoạn từ `i` **đến trước** `j`. Đoạn `a[i..j]` (tính cả `j`) viết là `a[i:j+1]`.
- `x[::-1]` đảo ngược list `x`.
- **Gán vào slice** `a[i:j+1] = ...` thay đoạn đó trong `a` (list thay đổi tại chỗ).

In [2]:
a = [10, 20, 30, 40, 50]
print(a[1:4])            # [20, 30, 40]
print(a[1:4][::-1])      # [40, 30, 20]
a[1:4] = a[1:4][::-1]
print(a)                 # [10, 40, 30, 20, 50]

[20, 30, 40]
[40, 30, 20]
[10, 40, 30, 20, 50]


### 2.2. Hoán đổi hai phần tử: `a[i], a[j] = a[j], a[i]`

Python tính vế phải trước (tạo tuple `(a[j], a[i])`) rồi gán lần lượt — không cần biến tạm. Đảo đoạn độ dài 2 `a[i:i+2]` chính là hoán đổi `a[i]` và `a[i+1]`.

In [3]:
a = [1, 2, 3]
a[0], a[1] = a[1], a[0]
print(a)                    # [2, 1, 3]
b_ = [1, 2, 3]
b_[0:2] = b_[0:2][::-1]
print(b_, a == b_)          # cùng kết quả

[2, 1, 3]
[2, 1, 3] True


### 2.3. `sorted` và so sánh list

`sorted(x)` trả **list mới** đã sắp xếp (không đổi `x`); `x.sort()` sắp xếp **tại chỗ** và trả `None`. Hai list bằng nhau `==` khi cùng độ dài và từng vị trí bằng nhau.

In [4]:
x = [3, 1, 2]
print(sorted(x), x)             # [1, 2, 3] [3, 1, 2]
print(x.sort(), x)              # None [1, 2, 3]
print([1, 2, 3] == [1, 2, 3], [1, 2] == [2, 1])

[1, 2, 3] [3, 1, 2]
None [1, 2, 3]
True False


### 2.4. `Counter` — đếm số lần xuất hiện

`Counter(list)` (từ `collections`) tạo dict `giá_trị → số_lần`. Hai `Counter` so sánh `==` bằng nhau khi mọi giá trị có cùng số lần. Đây là cách biểu diễn **multiset** (tập có lặp).

In [5]:
from collections import Counter
print(Counter([1, 2, 2, 3]))
print(Counter([2, 3, 2, 1]) == Counter([1, 2, 2, 3]))   # True — thứ tự không quan trọng
print(Counter([1, 2, 2]) == Counter([1, 1, 2]))         # False — số lần khác

Counter({2: 2, 1: 1, 3: 1})
True
False


### 2.5. Bất biến (invariant) của một thao tác

Một **bất biến** là đại lượng **không đổi** dù ta làm thao tác bao nhiêu lần. Muốn chứng minh "không thể biến A thành B", cách kinh điển: tìm bất biến mà A và B có **giá trị khác nhau**.

Với thao tác đảo đoạn: phần tử chỉ **đổi chỗ**, không thêm/bớt → `Counter(arr)` là bất biến. Kiểm chứng bằng nhiều lần đảo ngẫu nhiên:

In [6]:
import random
rng = random.Random(1460)
a = [5, 1, 5, 2, 9, 1]
before = Counter(a)
for _ in range(1000):
    i = rng.randrange(len(a)); j = rng.randrange(i, len(a))
    a[i:j + 1] = a[i:j + 1][::-1]
print("sau 1000 lần đảo:", a)
print("Counter không đổi?", Counter(a) == before)

sau 1000 lần đảo: [2, 5, 9, 1, 1, 5]
Counter không đổi? True


## 3. Cách 1 — Mô phỏng: thật sự biến `arr` thành `target`

Cách nghĩ trực tiếp: hãy **xây dựng** dãy thao tác. Giống selection sort:
- Với vị trí `i = 0, 1, ..., n-1`: tìm trong `arr[i:]` một vị trí `j` có `arr[j] == target[i]`.
  - Không tìm thấy → không thể (thiếu giá trị) → `False`.
  - Tìm thấy → đảo đoạn `arr[i..j]`: phần tử đó nhảy về vị trí `i`. Các vị trí `< i` không bị động tới.
- Xong vòng → `arr == target` → `True`.

In [7]:
def can_be_equal_simulate(target, arr):
    arr = arr[:]                         # sao chép để không sửa input của người gọi
    n = len(arr)
    for i in range(n):
        j = i
        while j < n and arr[j] != target[i]:
            j += 1
        if j == n:
            return False
        arr[i:j + 1] = arr[i:j + 1][::-1]
    return True

print(can_be_equal_simulate([1, 2, 3, 4], [2, 4, 1, 3]))   # True
print(can_be_equal_simulate([7], [7]))                     # True
print(can_be_equal_simulate([3, 7, 9], [3, 7, 11]))        # False

True
True
False


`arr[:]` tạo **bản sao nông** của list. Nếu không sao chép, hàm sẽ làm thay đổi list mà người gọi truyền vào — một "tác dụng phụ" khó chịu.

### 3.1. Trace cách 1 — thấy từng phép đảo

In [8]:
def trace_simulate(target, arr):
    arr = arr[:]
    n = len(arr)
    print("target =", target, "| arr =", arr)
    for i in range(n):
        j = i
        while j < n and arr[j] != target[i]:
            j += 1
        if j == n:
            print(f"  i={i}: không còn giá trị {target[i]} trong arr[{i}:] → False")
            return False
        arr[i:j + 1] = arr[i:j + 1][::-1]
        print(f"  i={i}: cần {target[i]}, thấy ở j={j}, đảo arr[{i}..{j}] → {arr}")
    print("  arr == target → True")
    return True

trace_simulate([1, 2, 3, 4], [2, 4, 1, 3])
print()
trace_simulate([3, 7, 9], [3, 7, 11])

target = [1, 2, 3, 4] | arr = [2, 4, 1, 3]
  i=0: cần 1, thấy ở j=2, đảo arr[0..2] → [1, 4, 2, 3]
  i=1: cần 2, thấy ở j=2, đảo arr[1..2] → [1, 2, 4, 3]
  i=2: cần 3, thấy ở j=3, đảo arr[2..3] → [1, 2, 3, 4]
  i=3: cần 4, thấy ở j=3, đảo arr[3..3] → [1, 2, 3, 4]
  arr == target → True

target = [3, 7, 9] | arr = [3, 7, 11]
  i=0: cần 3, thấy ở j=0, đảo arr[0..0] → [3, 7, 11]
  i=1: cần 7, thấy ở j=1, đảo arr[1..1] → [3, 7, 11]
  i=2: không còn giá trị 9 trong arr[2:] → False


False

Khi `j == i` (phần tử đã đúng chỗ), "đảo" đoạn độ dài 1 không làm gì — hợp lệ vì đề cho phép mảng con khác rỗng bất kỳ.

### 3.2. Chi phí của mô phỏng

Mỗi `i`: tìm `j` tốn `O(n)`, đảo tốn `O(n)` → tổng `O(n²)`. Với `n = 1000` là ~10⁶ phép, vẫn qua, nhưng ta đang **làm thật** một việc mà đề chỉ hỏi **có/không**.

## 4. Nút thắt và insight

**Nút thắt:** đề chỉ hỏi "có thể không?", ta không cần dãy thao tác cụ thể.

**Insight (hai chiều):**
1. **Điều kiện cần:** đảo không đổi multiset (mục 2.5). Nếu `Counter(arr) != Counter(target)` → chắc chắn `False`.
2. **Điều kiện đủ:** nếu cùng multiset, cách 1 **luôn** tìm được `j` ở mỗi bước (vì `arr[i:]` và `target[i:]` luôn cùng multiset — xem mục 10), nên luôn thành công. Hoặc nhìn đơn giản hơn: đảo đoạn độ dài 2 = hoán đổi kề nhau, mà **bubble sort** chỉ dùng hoán đổi kề nhau để sắp xếp bất kỳ dãy nào.

> Kết luận: `arr` biến được thành `target` ⇔ hai mảng có **cùng các phần tử với cùng số lần xuất hiện**.

Bài "đảo mảng con" thực chất là bài **so sánh multiset** (giống kiểm tra anagram).

In [9]:
for t, a in [([1, 2, 3, 4], [2, 4, 1, 3]), ([3, 7, 9], [3, 7, 11]), ([1, 1, 2], [1, 2, 2])]:
    print(t, a, "| cùng multiset?", Counter(t) == Counter(a), "| mô phỏng:", can_be_equal_simulate(t, a))

[1, 2, 3, 4] [2, 4, 1, 3] | cùng multiset? True | mô phỏng: True
[3, 7, 9] [3, 7, 11] | cùng multiset? False | mô phỏng: False
[1, 1, 2] [1, 2, 2] | cùng multiset? False | mô phỏng: False


Ví dụ cuối `[1,1,2]` vs `[1,2,2]`: **cùng tập giá trị** `{1, 2}` nhưng **khác số lần** → `False`. Vì vậy phải so multiset, không phải `set`.

## 5. Cách 2 — Sắp xếp rồi so sánh (theo hint)

Hai mảng cùng multiset ⇔ sau khi sắp xếp chúng giống hệt nhau.

In [10]:
def can_be_equal_sort(target, arr):
    return sorted(target) == sorted(arr)

print(can_be_equal_sort([1, 2, 3, 4], [2, 4, 1, 3]), can_be_equal_sort([7], [7]), can_be_equal_sort([3, 7, 9], [3, 7, 11]))
print(sorted([3, 7, 9]), sorted([3, 7, 11]))

True True False
[3, 7, 9] [3, 7, 11]


## 6. Cách 3 (lời giải chính) — `Counter`

### 6.1. Mổ xẻ

```python
return Counter(target) == Counter(arr)
```

- `Counter(target)`: một lượt `O(n)` đếm từng giá trị.
- `Counter(arr)`: tương tự.
- `==`: so từng key/số lần — `O(số giá trị khác nhau)`.

Viết "thủ công" để thấy rõ: đếm `target` cộng, đếm `arr` trừ; cuối cùng mọi số đếm phải bằng 0.

In [11]:
def can_be_equal_counter(target, arr):
    return Counter(target) == Counter(arr)

def can_be_equal_balance(target, arr):
    balance = {}
    for x in target:
        balance[x] = balance.get(x, 0) + 1
    for x in arr:
        balance[x] = balance.get(x, 0) - 1
        if balance[x] < 0:              # arr có x nhiều hơn target → dừng sớm
            return False
    return all(v == 0 for v in balance.values())

print(can_be_equal_counter([1, 2, 3, 4], [2, 4, 1, 3]), can_be_equal_balance([1, 2, 3, 4], [2, 4, 1, 3]))
print(can_be_equal_counter([3, 7, 9], [3, 7, 11]), can_be_equal_balance([3, 7, 9], [3, 7, 11]))

True True
False False


`balance.get(x, 0)` đọc giá trị của `x`, trả `0` nếu chưa có key — tránh `KeyError`.

## 7. TRACE bảng cân bằng — `target = [1,2,2,3]`, `arr = [2,3,1,2]`

| Bước | phần tử | thao tác | `balance` sau |
|---|---:|---|---|
| target | 1 | +1 | {1:1} |
| target | 2 | +1 | {1:1, 2:1} |
| target | 2 | +1 | {1:1, 2:2} |
| target | 3 | +1 | {1:1, 2:2, 3:1} |
| arr | 2 | −1 | {1:1, 2:1, 3:1} |
| arr | 3 | −1 | {1:1, 2:1, 3:0} |
| arr | 1 | −1 | {1:0, 2:1, 3:0} |
| arr | 2 | −1 | {1:0, 2:0, 3:0} → mọi giá trị 0 → **True** |

In [12]:
def trace_balance(target, arr):
    balance = {}
    for x in target:
        balance[x] = balance.get(x, 0) + 1
        print(f"target +{x}: {balance}")
    for x in arr:
        balance[x] = balance.get(x, 0) - 1
        print(f"arr    -{x}: {balance}")
        if balance[x] < 0:
            print(f"  balance[{x}] < 0 → arr có {x} nhiều hơn target → False")
            return False
    ok = all(v == 0 for v in balance.values())
    print("mọi số đếm bằng 0?", ok)
    return ok

trace_balance([1, 2, 2, 3], [2, 3, 1, 2])
print()
trace_balance([3, 7, 9], [3, 7, 11])

target +1: {1: 1}
target +2: {1: 1, 2: 1}
target +2: {1: 1, 2: 2}
target +3: {1: 1, 2: 2, 3: 1}
arr    -2: {1: 1, 2: 1, 3: 1}
arr    -3: {1: 1, 2: 1, 3: 0}
arr    -1: {1: 0, 2: 1, 3: 0}
arr    -2: {1: 0, 2: 0, 3: 0}
mọi số đếm bằng 0? True

target +3: {3: 1}
target +7: {3: 1, 7: 1}
target +9: {3: 1, 7: 1, 9: 1}
arr    -3: {3: 0, 7: 1, 9: 1}
arr    -7: {3: 0, 7: 0, 9: 1}
arr    -11: {3: 0, 7: 0, 9: 1, 11: -1}
  balance[11] < 0 → arr có 11 nhiều hơn target → False


False

Ở ví dụ 3, `11` không có trong `target` nên `balance[11] = -1 < 0` → dừng ngay.

## 8. Cách 4 — Mảng đếm kích thước cố định

Vì `1 <= giá trị <= 1000`, dùng list `cnt` dài 1001 thay cho dict: `cnt[x] += 1` với `target`, `cnt[x] -= 1` với `arr`. Không cần băm, chỉ cần chỉ số.

In [13]:
def can_be_equal_array(target, arr):
    cnt = [0] * 1001
    for x in target:
        cnt[x] += 1
    for x in arr:
        cnt[x] -= 1
    return not any(cnt)          # any(cnt) True nếu có số khác 0

print(can_be_equal_array([1, 2, 3, 4], [2, 4, 1, 3]), can_be_equal_array([3, 7, 9], [3, 7, 11]))
print(any([0, 0, 0]), any([0, -1, 0]))

True False
False True


`any(x)` trả `True` nếu **có ít nhất một** phần tử "truthy" (số khác 0 là truthy). `not any(cnt)` ⇔ mọi phần tử bằng 0.

## 9. Code nộp LeetCode (Python 3)

In [14]:
from collections import Counter

class Solution:
    def canBeEqual(self, target: list[int], arr: list[int]) -> bool:
        return Counter(target) == Counter(arr)

sol = Solution()
print(sol.canBeEqual([1, 2, 3, 4], [2, 4, 1, 3]))   # True
print(sol.canBeEqual([7], [7]))                     # True
print(sol.canBeEqual([3, 7, 9], [3, 7, 11]))        # False

True
True
False


Bản theo hint (cũng Accepted):

```python
class Solution:
    def canBeEqual(self, target: list[int], arr: list[int]) -> bool:
        return sorted(target) == sorted(arr)
```

## 10. Vì sao thuật toán đúng?

**(⇒) Nếu biến được thì cùng multiset.** Mỗi lần đảo chỉ hoán vị các phần tử trong một đoạn, không thêm/bớt giá trị → `Counter(arr)` không đổi qua mọi bước. Nếu cuối cùng `arr == target` thì `Counter(arr_ban_đầu) = Counter(target)`.

**(⇐) Nếu cùng multiset thì biến được** — chứng minh bằng thuật toán cách 1.
**Invariant:** trước bước `i`, `arr[0..i-1] == target[0..i-1]` và `arr[i:]`, `target[i:]` có cùng multiset.
- Khởi tạo `i = 0`: phần đầu rỗng; phần sau là toàn mảng, cùng multiset theo giả thiết. ✓
- Duy trì: vì `arr[i:]` cùng multiset với `target[i:]` mà `target[i]` thuộc `target[i:]`, nên có `j >= i` với `arr[j] = target[i]` — **luôn tìm thấy**. Đảo `arr[i..j]` đưa giá trị đó về `i`, không chạm `arr[0..i-1]`, và chỉ hoán vị trong `arr[i:]`. Bỏ phần tử bằng nhau ở vị trí `i` của cả hai → `arr[i+1:]`, `target[i+1:]` vẫn cùng multiset. ✓
- Kết thúc `i = n`: `arr == target`. ✓

Vậy điều kiện "cùng multiset" là **cần và đủ**, và cả 4 cách đều kiểm tra đúng điều kiện đó.

## 11. Phân tích Time / Space

Gọi `n = len(arr)`, `V = 1000` (giá trị lớn nhất).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng | `O(n²)` | `O(n)` (bản sao) | có thể in ra dãy thao tác |
| 2. Sort + so sánh | `O(n log n)` | `O(n)` | theo hint |
| 3. `Counter` | `O(n)` trung bình | `O(k)` với `k` = số giá trị khác nhau | **khuyên dùng** |
| 4. Mảng đếm | `O(n + V)` | `O(V)` | không băm, dựa vào giới hạn giá trị |

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.canBeEqual([1, 2, 3, 4], [2, 4, 1, 3]), True)
check("ví dụ 2", sol.canBeEqual([7], [7]), True)
check("ví dụ 3", sol.canBeEqual([3, 7, 9], [3, 7, 11]), False)
check("cùng set, khác số lần", sol.canBeEqual([1, 1, 2], [1, 2, 2]), False)
check("đảo cả mảng", sol.canBeEqual([1, 2, 3], [3, 2, 1]), True)
check("có trùng lặp, cùng multiset", sol.canBeEqual([5, 5, 1, 5], [1, 5, 5, 5]), True)
check("một phần tử khác", sol.canBeEqual([1], [2]), False)
check("giá trị biên 1 và 1000", sol.canBeEqual([1, 1000], [1000, 1]), True)

PASS  ví dụ 1: True
PASS  ví dụ 2: True
PASS  ví dụ 3: False
PASS  cùng set, khác số lần: False
PASS  đảo cả mảng: True
PASS  có trùng lặp, cùng multiset: True
PASS  một phần tử khác: False
PASS  giá trị biên 1 và 1000: True


### 12.1. Đối chiếu 4 cách + kiểm chứng bằng BFS vét cạn

Với mảng ngắn (n ≤ 5), ta có thể **thật sự thử mọi dãy đảo** bằng BFS (duyệt theo chiều rộng) trên các trạng thái — một oracle không dựa trên lý thuyết multiset. `deque` là hàng đợi hai đầu (`popleft` `O(1)`), `tuple` dùng làm phần tử của `set` vì list không băm được.

In [16]:
from collections import deque
from itertools import product

def can_be_equal_bfs(target, arr):
    start, goal = tuple(arr), tuple(target)
    seen, q = {start}, deque([start])
    while q:
        cur = q.popleft()
        if cur == goal:
            return True
        for i in range(len(cur)):
            for j in range(i + 1, len(cur)):
                nxt = cur[:i] + cur[i:j + 1][::-1] + cur[j + 1:]
                if nxt not in seen:
                    seen.add(nxt); q.append(nxt)
    return False

methods = [can_be_equal_simulate, can_be_equal_sort, can_be_equal_counter,
           can_be_equal_balance, can_be_equal_array, Solution().canBeEqual]
count = 0
for n in range(1, 5):
    for t in product([1, 2, 3], repeat=n):
        for a in product([1, 2, 3], repeat=n):
            exp = can_be_equal_bfs(list(t), list(a))
            for f in methods:
                assert f(list(t), list(a)) == exp, (f.__name__, t, a)
            count += 1
print("Đã kiểm tra", count, "cặp mảng (n ≤ 4, giá trị 1..3): mọi cách khớp BFS")

Đã kiểm tra 7380 cặp mảng (n ≤ 4, giá trị 1..3): mọi cách khớp BFS


In [17]:
rng = random.Random(14600)
for _ in range(300):
    n = rng.randint(1, 1000)
    t = [rng.randint(1, 1000) for _ in range(n)]
    a = t[:]; rng.shuffle(a)
    if rng.random() < 0.5:
        a[rng.randrange(n)] = rng.randint(1, 1000)      # có thể làm lệch multiset
    exp = can_be_equal_sort(t, a)
    assert can_be_equal_counter(t, a) == can_be_equal_array(t, a) == can_be_equal_balance(t, a) == exp
print("300 mảng lớn ngẫu nhiên: PASS")

300 mảng lớn ngẫu nhiên: PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1 — So sánh bằng `set` (bỏ mất số lần xuất hiện)

In [18]:
def wrong_set(target, arr):
    return set(target) == set(arr)

print("Sai:", wrong_set([1, 1, 2], [1, 2, 2]), "| Đúng:", can_be_equal_counter([1, 1, 2], [1, 2, 2]))

Sai: True | Đúng: False


**Sửa:** `Counter` hoặc `sorted` — giữ cả số lần.

### Lỗi 2 — Dùng `.sort()` trong biểu thức so sánh

In [19]:
def wrong_sort_inplace(target, arr):
    return target.sort() == arr.sort()     # None == None → luôn True!

print("Sai:", wrong_sort_inplace([3, 7, 9], [3, 7, 11]), "| Đúng:", can_be_equal_sort([3, 7, 9], [3, 7, 11]))

Sai: True | Đúng: False


**Vì sao sai:** `.sort()` trả `None`. Còn tệ hơn: nó **sửa** list gốc của người gọi. **Sửa:** `sorted(target) == sorted(arr)`.

### Lỗi 3 — Chỉ kiểm tra "mỗi phần tử của `arr` có trong `target`"

In [20]:
def wrong_membership(target, arr):
    return all(x in target for x in arr)

print("Sai:", wrong_membership([1, 2, 3], [1, 1, 1]), "| Đúng:", can_be_equal_counter([1, 2, 3], [1, 1, 1]))

Sai: True | Đúng: False


**Vì sao sai:** không đếm số lần, và `x in list` còn tốn `O(n)` mỗi lần → `O(n²)`.

### Lỗi 4 — Cố gắng mô phỏng mà sửa luôn input

Viết cách 1 nhưng quên `arr = arr[:]` → mảng của người gọi bị đổi. Trên LeetCode thường không sao, nhưng trong code thực tế (và khi test nhiều hàm trên cùng dữ liệu như notebook này) sẽ gây lỗi khó tìm.

In [21]:
def simulate_no_copy(target, arr):
    n = len(arr)
    for i in range(n):
        j = i
        while j < n and arr[j] != target[i]:
            j += 1
        if j == n:
            return False
        arr[i:j + 1] = arr[i:j + 1][::-1]
    return True

data = [2, 4, 1, 3]
simulate_no_copy([1, 2, 3, 4], data)
print("data sau khi gọi:", data, "← đã bị thay đổi!")

data sau khi gọi: [1, 2, 3, 4] ← đã bị thay đổi!


## 14. Mẹo nhận dạng pattern

- Đề cho một **thao tác** và hỏi "có thể biến A thành B không?" → tìm **bất biến** của thao tác (điều kiện cần) rồi chứng minh nó **đủ** bằng một cách xây dựng.
- "Đảo đoạn bất kỳ" / "hoán đổi tùy ý" / "sắp xếp lại tùy ý" → chỉ còn lại câu hỏi **multiset có bằng nhau không** → `Counter` hoặc `sorted`.
- Nếu thao tác bị **hạn chế** (chỉ đảo độ dài `k` cố định, chỉ hoán đổi vị trí cách nhau `k`…) thì bất biến mạnh hơn — xem biến thể.
- Bài cùng họ: *Valid Anagram* (242), *Determine if Two Strings Are Close* (1657), *Check if One String Swap Can Make Strings Equal* (1790), *Pancake Sorting* (969).

## 15. Biến thể

### 15.1. Valid Anagram (LeetCode 242) — cùng bản chất trên chuỗi

In [22]:
def is_anagram(s, t):
    return Counter(s) == Counter(t)

print(is_anagram("anagram", "nagaram"), is_anagram("rat", "car"))

True False


### 15.2. Đếm số phép đảo cách 1 thực sự dùng

Cách 1 dùng tối đa `n - 1` phép đảo **có tác dụng** (bỏ qua các lần `j == i`). Ở bước cuối `i = n-1`, phần tử luôn đã đúng chỗ.

In [23]:
def count_reversals(target, arr):
    arr = arr[:]
    ops = 0
    for i in range(len(arr)):
        j = arr.index(target[i], i)       # tìm từ vị trí i trở đi; ValueError nếu không có
        if j != i:
            arr[i:j + 1] = arr[i:j + 1][::-1]
            ops += 1
    return ops

print(count_reversals([1, 2, 3, 4], [2, 4, 1, 3]))   # 3
print(count_reversals([1, 2, 3, 4], [4, 3, 2, 1]))   # 1 — đảo cả mảng

3
1


`list.index(value, start)` trả vị trí đầu tiên `>= start` có giá trị `value`.

### 15.3. Chỉ được hoán đổi hai phần tử **cách nhau đúng 2** (đảo đoạn độ dài 3)

Đảo `[a, b, c]` → `[c, b, a]`: phần tử giữa đứng yên, hai đầu đổi chỗ — chỉ di chuyển giữa các vị trí **cùng tính chẵn lẻ**. Bất biến: multiset ở vị trí chẵn và multiset ở vị trí lẻ.

In [24]:
def can_equal_len3(target, arr):
    return Counter(target[0::2]) == Counter(arr[0::2]) and Counter(target[1::2]) == Counter(arr[1::2])

print(can_equal_len3([1, 2, 3, 4], [3, 2, 1, 4]))   # True
print(can_equal_len3([1, 2, 3, 4], [2, 1, 3, 4]))   # False — 1 và 2 khác tính chẵn lẻ

True
False


`a[0::2]` lấy phần tử ở vị trí chẵn (bước 2 từ 0), `a[1::2]` lấy vị trí lẻ.

## 16. Bài tập luyện tập (tăng dần)

**Bài 1 — Tính tay.** `target = [4,1,2]`, `arr = [2,4,1]`. Mô phỏng cách 1: ghi từng `i`, `j`, mảng sau khi đảo.

**Bài 2 — Dự đoán output.**
```python
a = [1, 2, 3, 4, 5]
a[1:4] = a[1:4][::-1]
print(a, a[::2], sorted([3, 1, 2]) == [1, 2, 3])
```

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết lời giải đếm bằng dict (không dùng `Counter`)

In [25]:
def exercise_can_be_equal(target, arr):
    # TODO 1: đếm target vào dict (+1)
    # TODO 2: trừ theo arr (-1), nếu âm thì False
    # TODO 3: kiểm tra mọi số đếm bằng 0
    pass

# assert exercise_can_be_equal([1,2,3,4], [2,4,1,3]) == True
# assert exercise_can_be_equal([1,1,2], [1,2,2]) == False

### Bài 4 — TODO: trả về dãy thao tác
Trả list các cặp `(i, j)` (đảo `arr[i..j]`, `i < j`) biến `arr` thành `target`, hoặc `None` nếu không thể.

In [26]:
def exercise_reversal_steps(target, arr):
    # TODO: dựa trên cách 1, ghi lại (i, j) mỗi khi j != i
    pass

# steps = exercise_reversal_steps([1,2,3,4], [2,4,1,3])
# (tự áp dụng steps lên [2,4,1,3] và kiểm tra ra [1,2,3,4])
# assert exercise_reversal_steps([3,7,9], [3,7,11]) is None

### Bài 5 — TODO: Check if One String Swap Can Make Strings Equal (LeetCode 1790)
Trả `True` nếu có thể làm `s1 == s2` bằng **tối đa một** lần hoán đổi hai ký tự trong **một** chuỗi.

In [27]:
def exercise_one_swap(s1, s2):
    # TODO: thu thập các vị trí khác nhau; phải là 0 vị trí, hoặc đúng 2 vị trí "chéo nhau"
    pass

# assert exercise_one_swap("bank", "kanb") == True
# assert exercise_one_swap("attack", "defend") == False
# assert exercise_one_swap("kelb", "kelb") == True
# assert exercise_one_swap("abcd", "dcba") == False

### Bài 6 — Mini challenge: chỉ được đảo đoạn có độ dài **đúng `k`**
Cho `k >= 1`. Viết hàm kiểm tra bằng BFS cho mảng nhỏ (n ≤ 6), rồi quan sát và đoán quy luật cho `k = 2` và `k = 3`.

In [28]:
def exercise_reverse_len_k(target, arr, k):
    # TODO: BFS như mục 12.1 nhưng chỉ đảo các đoạn cur[i:i+k]
    pass

# assert exercise_reverse_len_k([1,2,3], [3,2,1], 3) == True
# assert exercise_reverse_len_k([1,2,3], [2,1,3], 3) == False
# assert exercise_reverse_len_k([1,2,3], [2,3,1], 2) == True

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Cân bằng số lần: + cho target, − cho arr. | `dict` + `get(x, 0)`. | xem `can_be_equal_balance` — tự viết lại không nhìn |
| 4 | Cách 1 chính là lời giải xây dựng. | vòng `i`, tìm `j`, ghi `(i, j)`. | `if j == n: return None`; `if j != i: steps.append((i, j))` |
| 5 | Tìm mọi chỉ số `p` với `s1[p] != s2[p]`. | 0 vị trí → True; 2 vị trí `p, q` → kiểm tra chéo. | `s1[p] == s2[q] and s1[q] == s2[p]` |
| 6 | Trạng thái = tuple, cạnh = một lần đảo. | `deque`, `seen`. | `nxt = cur[:i] + cur[i:i+k][::-1] + cur[i+k:]`; quy luật: `k=2` ⇔ cùng multiset; `k=3` ⇔ multiset chẵn/lẻ (n ≥ 3) |

## 18. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** `i=0`: cần 4, ở `j=1`, đảo `[2,4]` → `[4,2,1]`; `i=1`: cần 1, ở `j=2`, đảo `[2,1]` → `[4,1,2]`; `i=2`: đã đúng → **True**.
**Bài 2:** `[1, 4, 3, 2, 5] [1, 3, 5] True`.

In [29]:
def answer_can_be_equal(target, arr):
    cnt = {}
    for x in target:
        cnt[x] = cnt.get(x, 0) + 1
    for x in arr:
        cnt[x] = cnt.get(x, 0) - 1
        if cnt[x] < 0:
            return False
    return all(v == 0 for v in cnt.values())

def answer_reversal_steps(target, arr):
    arr = arr[:]
    n, steps = len(arr), []
    for i in range(n):
        j = i
        while j < n and arr[j] != target[i]:
            j += 1
        if j == n:
            return None
        if j != i:
            arr[i:j + 1] = arr[i:j + 1][::-1]
            steps.append((i, j))
    return steps

def answer_one_swap(s1, s2):
    diff = [p for p in range(len(s1)) if s1[p] != s2[p]]
    if not diff:
        return True
    if len(diff) != 2:
        return False
    p, q = diff
    return s1[p] == s2[q] and s1[q] == s2[p]

def answer_reverse_len_k(target, arr, k):
    start, goal = tuple(arr), tuple(target)
    seen, q = {start}, deque([start])
    while q:
        cur = q.popleft()
        if cur == goal:
            return True
        for i in range(len(cur) - k + 1):
            nxt = cur[:i] + cur[i:i + k][::-1] + cur[i + k:]
            if nxt not in seen:
                seen.add(nxt); q.append(nxt)
    return False

assert answer_can_be_equal([1,2,3,4], [2,4,1,3]) == True
assert answer_can_be_equal([1,1,2], [1,2,2]) == False
steps = answer_reversal_steps([1,2,3,4], [2,4,1,3])
a = [2,4,1,3]
for i, j in steps:
    a[i:j + 1] = a[i:j + 1][::-1]
assert a == [1,2,3,4], (steps, a)
assert answer_reversal_steps([3,7,9], [3,7,11]) is None
assert answer_one_swap("bank", "kanb") == True
assert answer_one_swap("attack", "defend") == False
assert answer_one_swap("kelb", "kelb") == True
assert answer_one_swap("abcd", "dcba") == False
assert answer_reverse_len_k([1,2,3], [3,2,1], 3) == True
assert answer_reverse_len_k([1,2,3], [2,1,3], 3) == False
assert answer_reverse_len_k([1,2,3], [2,3,1], 2) == True
print("steps bài 4:", steps)
print("Bài 1:", trace_simulate([4, 1, 2], [2, 4, 1]))
a = [1, 2, 3, 4, 5]; a[1:4] = a[1:4][::-1]
print("Bài 2:", a, a[::2], sorted([3, 1, 2]) == [1, 2, 3])
print("Đáp án bài 3–6: PASS")

steps bài 4: [(0, 2), (1, 2), (2, 3)]
target = [4, 1, 2] | arr = [2, 4, 1]
  i=0: cần 4, thấy ở j=1, đảo arr[0..1] → [4, 2, 1]
  i=1: cần 1, thấy ở j=2, đảo arr[1..2] → [4, 1, 2]
  i=2: cần 2, thấy ở j=2, đảo arr[2..2] → [4, 1, 2]
  arr == target → True
Bài 1: True
Bài 2: [1, 4, 3, 2, 5] [1, 3, 5] True
Đáp án bài 3–6: PASS


In [30]:
# Kiểm chứng quy luật bài 6 trên mọi hoán vị nhỏ (giá trị phân biệt)
from itertools import permutations
for n in range(3, 6):
    base = list(range(n))
    for perm in permutations(base):
        p = list(perm)
        assert answer_reverse_len_k(base, p, 2) == can_be_equal_counter(base, p)
        assert answer_reverse_len_k(base, p, 3) == can_equal_len3(base, p)
print("Quy luật k=2 (cùng multiset) và k=3 (multiset chẵn/lẻ) đúng với mọi hoán vị n=3..5")

Quy luật k=2 (cùng multiset) và k=3 (multiset chẵn/lẻ) đúng với mọi hoán vị n=3..5


## 19. Template nhớ nhanh + checklist submit

```python
# "Có thể biến A thành B bằng thao tác X không?"
# 1) Tìm bất biến của X  →  điều kiện cần
# 2) Chứng minh đủ bằng cách xây dựng (thường là sắp xếp từng vị trí)
# Với X = đảo mảng con bất kỳ: bất biến = multiset
from collections import Counter
return Counter(target) == Counter(arr)        # hoặc sorted(target) == sorted(arr)
```

**Checklist trước khi nộp**
- [ ] So **multiset** (`Counter`/`sorted`), không phải `set`.
- [ ] Dùng `sorted(...)`, không dùng `.sort()` trong biểu thức (trả `None`).
- [ ] Hiểu vì sao điều kiện là **đủ** (đảo độ dài 2 = hoán đổi kề nhau).
- [ ] Không sửa input nếu mô phỏng (`arr[:]`).
- [ ] Method tên `canBeEqual`, trả về `bool`.